In [ ]:
'''
problems.csv를 한 행씩 훑으면서, 각 행의 id와 동일한 problem_id를 가진 problem_solutions.csv의 행들(solutions)과 problem_testcases.csv의 행들(testcases)을 뽑아. 
그 후, 각 solution에 대해(problem_solutions.csv의 solution 열값. 파이썬 코드임) 
'./3) execute/{problem_id}-{solution_order}.csv'를 만들고 그 안에 
problem_id, solution_order, testcase_order, input, output, error, result, time(s), memory(kb), validation 열을 만들어. 
각 행에는 problem_id, solution_order(여기까진 파일명과 모두 똑같은 값일 것), testcase_order(1~해당 문제의 테스트케이스 개수), 
input(testcase의 input열값), output(output열값), error, result(solution을 실행한 후 testcase의 input을 입력값으로 넣은 결과 출력값. 
만약 시간초과(10초 이상 소요) 혹은 런타임/문법 에러 시에는 result=null, error=true. 아닌 경우라면 error=false), 
time(해당 testcase에 대해 solution을 실행하는 데에 걸린 소요시간, seconds 단위. error=false인 경우에만 기입하고 아니면 null),
memory(해당 testcase에 대해 solution을 실행하는 데에 사용한 메모리, kb 단위. error=false인 경우에만 기입하고 아니면 null), 
validation(error=true라면 validation=false, error=false라면 output == result여야만 validation=true (문자열 검사 시 trim으로 양옆 공백 자르기) 값이 들어가도록 하면 돼). 
특정 문제의 특정 솔루션 번호에 대한 csv가 이미 있다면 건너뛰도록 작성해줘. 
각 csv는 solution 하나를 완전히 검토한 다음에 저장해줘. 
즉, 예를 들어 3-5.csv가 존재한다는 건 3번 문제의 모든 테스트케이스를 5번 solution으로 실행해봤다는 의미가 되는 거야.

그리고 각 문제/solution을 진행하기 전/후에는 print로 상황을 출력해서 진행도를 알 수 있게 해줘. 
특정 문제 검수를 시작하면 그게 'problems.csv의 총 몇 문제 중에 몇 번째인지, 그리고 id는 몇 번인지'를 출력하고, 
특정 solution을 실행한다면 그게 '해당 문제의 총 몇 개의 solution 중에 몇 번째(solution_order)인지'를 출력하고, 
특정 테스트케이스를 실행한다면 그게 '해당 문제의 총 몇 개의 테스트케이스 중에 몇 번째(testcase_order)인지'를 출력하는 거야. 
그리고 각 testcase를 실행하는 데 걸린 시간도 출력해주고. 이미 csv가 존재해서 건너뛰는 경우도 무슨 파일을 건너뛰었다고 출력해주면 돼.
'''

In [1]:
import os
import sys
import time
import pandas as pd
import subprocess

# --- 1. 환경 설정 및 디렉토리 준비 ---
OUTPUT_DIR = './3) execute'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 메모리 측정을 위한 래퍼(Wrapper) 스크립트 생성
# subprocess 내에서 코드를 실행하고, resource 모듈로 메모리를 측정해 별도 파일에 저장합니다.
runner_code = """
import sys
import resource

try:
    with open(sys.argv[1], 'r', encoding='utf-8') as f:
        code = f.read()
    # 격리된 네임스페이스에서 코드 실행 (BOJ 표준 입출력 방식)
    exec(code, {'__name__': '__main__'})
except Exception:
    # 런타임/문법 에러 발생 시 비정상 종료
    sys.exit(1)
finally:
    try:
        # ru_maxrss는 리눅스 환경에서 KB 단위를 반환합니다.
        maxrss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        with open(sys.argv[2], 'w', encoding='utf-8') as f:
            f.write(str(maxrss))
    except Exception:
        pass
"""

with open('_runner.py', 'w', encoding='utf-8') as f:
    f.write(runner_code)

# --- 2. 데이터 불러오기 ---
print("CSV 파일들을 불러오는 중입니다. 잠시만 기다려주세요...")
df_problems = pd.read_csv('problems.csv')
df_solutions = pd.read_csv('problem_solutions.csv')
df_testcases = pd.read_csv('problem_testcases.csv')
print("데이터 로드 완료! 본격적인 검증을 시작합니다.\n")

total_problems = len(df_problems)

# --- 3. 메인 검증 파이프라인 ---
for p_idx, p_row in enumerate(df_problems.itertuples(), start=1):
    problem_id = p_row.id
    print(f"\n[진행도] problems.csv 총 {total_problems}문제 중 {p_idx}번째 시작 (Problem ID: {problem_id})")
    
    # 현재 문제에 해당하는 솔루션과 테스트케이스 추출
    sols_for_prob = df_solutions[df_solutions['problem_id'] == problem_id]
    tcs_for_prob = df_testcases[df_testcases['problem_id'] == problem_id]
    
    total_sols = len(sols_for_prob)
    total_tcs = len(tcs_for_prob)
    
    if total_sols == 0 or total_tcs == 0:
        print(f"  ⚠️ 해당 문제에 대한 솔루션이나 테스트케이스가 없어 건너뜁니다.")
        continue

    # 각 솔루션에 대해 검증 진행
    for s_idx, sol_row in enumerate(sols_for_prob.itertuples(), start=1):
        solution_order = sol_row.solution_order
        solution_code = sol_row.solution
        
        csv_path = f"{OUTPUT_DIR}/{problem_id}-{solution_order}.csv"
        
        # 파일이 이미 존재하면 건너뜀
        if os.path.exists(csv_path):
            print(f"  ⏭ [건너뜀] 이미 파일이 존재합니다: {csv_path}")
            continue
            
        print(f"  ▶ [Solution] 해당 문제의 총 {total_sols}개의 solution 중 {s_idx}번째 (solution_order: {solution_order}) 시작")
        
        # 솔루션 코드를 임시 파일로 저장
        with open('_temp_sol.py', 'w', encoding='utf-8') as f:
            f.write(str(solution_code))
            
        results_data = []
        
        # 각 테스트케이스 실행
        for tc_idx, tc_row in enumerate(tcs_for_prob.itertuples(), start=1):
            tc_order = tc_row.testcase_order
            # NaN 방지 및 문자열 보장
            input_val = str(tc_row.input) if pd.notna(tc_row.input) else ""
            output_val = str(tc_row.output) if pd.notna(tc_row.output) else ""
            
            # 임시 파일들 초기화 (입력값 세팅)
            with open('_temp_in.txt', 'w', encoding='utf-8') as f:
                f.write(input_val)
            if os.path.exists('_temp_mem.txt'): os.remove('_temp_mem.txt')
            if os.path.exists('_temp_out.txt'): os.remove('_temp_out.txt')
            
            start_time = time.time()
            error = False
            
            try:
                # Subprocess로 독립 실행 (타임아웃 10초 강제)
                with open('_temp_in.txt', 'r', encoding='utf-8') as f_in, \
                     open('_temp_out.txt', 'w', encoding='utf-8') as f_out:
                    
                    subprocess.run(
                        [sys.executable, '_runner.py', '_temp_sol.py', '_temp_mem.txt'],
                        stdin=f_in,
                        stdout=f_out,
                        stderr=subprocess.DEVNULL, # 메인 콘솔 오염 방지
                        timeout=10,
                        check=True
                    )
                elapsed_time = time.time() - start_time
            except subprocess.TimeoutExpired:
                elapsed_time = None
                error = True
            except subprocess.CalledProcessError:
                # 에러코드 반환 (런타임 에러, 문법 에러 등)
                elapsed_time = None
                error = True
                
            # 실행 결과 파싱 및 검증
            if error:
                result_val = None
                mem_kb = None
                validation = False
                print(f"    - [Testcase] 총 {total_tcs}개 중 {tc_idx}번째 (testcase_order: {tc_order}) | 결과: 에러/시간초과")
            else:
                with open('_temp_out.txt', 'r', encoding='utf-8', errors='ignore') as f:
                    result_val = f.read()
                    
                try:
                    with open('_temp_mem.txt', 'r', encoding='utf-8') as f:
                        mem_kb = float(f.read().strip())
                except:
                    mem_kb = None
                    
                # 공백 무시하고 문자열 일치 여부 확인
                validation = (output_val.strip() == result_val.strip())
                #print(f"    - [Testcase] 총 {total_tcs}개 중 {tc_idx}번째 (testcase_order: {tc_order}) | 소요시간: {elapsed_time:.3f}s, 메모리: {mem_kb}kb")
            
            # 결과 리스트에 행 추가
            results_data.append({
                'problem_id': problem_id,
                'solution_order': solution_order,
                'testcase_order': tc_order,
                'input': input_val,
                'output': output_val,
                'error': error,
                'result': result_val,
                'time(s)': elapsed_time,
                'memory(kb)': mem_kb,
                'validation': validation
            })
            
        # 해당 솔루션의 모든 테스트케이스가 끝나면 CSV로 저장
        df_result = pd.DataFrame(results_data)
        df_result.to_csv(csv_path, index=False, encoding='utf-8-sig')
        print(f"  ✔ [저장 완료] {csv_path} (총 {len(df_result)}개 테스트케이스 기록됨)")

# 검사 종료 후 임시 파일 정리
temp_files = ['_runner.py', '_temp_sol.py', '_temp_in.txt', '_temp_out.txt', '_temp_mem.txt']
for tf in temp_files:
    if os.path.exists(tf):
        os.remove(tf)

print("\n🎉 모든 파이프라인 처리가 완료되었습니다!")

CSV 파일들을 불러오는 중입니다. 잠시만 기다려주세요...
데이터 로드 완료! 본격적인 검증을 시작합니다.


[진행도] problems.csv 총 5394문제 중 1번째 시작 (Problem ID: 2)
  ▶ [Solution] 해당 문제의 총 10개의 solution 중 1번째 (solution_order: 1) 시작
  ✔ [저장 완료] ./3) execute/2-1.csv (총 94개 테스트케이스 기록됨)
  ▶ [Solution] 해당 문제의 총 10개의 solution 중 2번째 (solution_order: 2) 시작
  ✔ [저장 완료] ./3) execute/2-2.csv (총 94개 테스트케이스 기록됨)
  ▶ [Solution] 해당 문제의 총 10개의 solution 중 3번째 (solution_order: 3) 시작
  ✔ [저장 완료] ./3) execute/2-3.csv (총 94개 테스트케이스 기록됨)
  ▶ [Solution] 해당 문제의 총 10개의 solution 중 4번째 (solution_order: 4) 시작
  ✔ [저장 완료] ./3) execute/2-4.csv (총 94개 테스트케이스 기록됨)
  ▶ [Solution] 해당 문제의 총 10개의 solution 중 5번째 (solution_order: 5) 시작
  ✔ [저장 완료] ./3) execute/2-5.csv (총 94개 테스트케이스 기록됨)
  ▶ [Solution] 해당 문제의 총 10개의 solution 중 6번째 (solution_order: 6) 시작
  ✔ [저장 완료] ./3) execute/2-6.csv (총 94개 테스트케이스 기록됨)
  ▶ [Solution] 해당 문제의 총 10개의 solution 중 7번째 (solution_order: 7) 시작
  ✔ [저장 완료] ./3) execute/2-7.csv (총 94개 테스트케이스 기록됨)
  ▶ [Solution] 해당 문제의 총 10개의 solution 중 8번째 (soluti

KeyboardInterrupt: 